# Multi-night linking

**Goal:** connect one-night tracklets of the same object across nights days apart. The Minor Planet Center needs at least
two nights before an object can be designated. Read `docs/multi_night_linking.md` alongside this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from asthunt.config.load_config import load_config
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.models.load_model_bundle import load_model_bundle
from asthunt.pipeline.run_night_pipeline import run_night_pipeline
from asthunt.dataset.make_unit_rng import make_unit_rng
from asthunt.link.measure_sky_motion import measure_sky_motion
from asthunt.link.link_tracklets_across_nights import link_tracklets_across_nights

project = load_config("project"); paths = project["paths"]
configs = {name: load_config(name) for name in ["acquire", "inject", "detect", "link"]}
bundle = load_model_bundle(paths["models"], "v4")
device = "mps" if torch.backends.mps.is_available() else "cpu"

## 1. Search two nights three days apart (field 618, CCD 3, quadrant 4)

In [ ]:
motion = {}
for night in ["2019-01-20", "2019-01-23"]:
    unit = {"field": 618, "ccd": 3, "quadrant": 4, "night": night}
    result = run_night_pipeline(unit, load_unit_metadata(unit, paths["raw"]), configs, bundle, paths,
                                make_unit_rng(project["seed"], unit), device, write_reports=False)
    accepted = result["tracklets"][result["tracklets"]["tracklet_score"] >= bundle["thresholds"]["tracklet"]]
    members = result["members"][result["members"]["tracklet_id"].isin(accepted["tracklet_id"])]
    motion[night] = measure_sky_motion(members, result["detections"]).merge(accepted[["tracklet_id", "known_name"]], on="tracklet_id")
    print(f"{night}: {len(accepted)} accepted tracklets, {int((accepted['known_name'] != '').sum())} known")
first, second = motion["2019-01-20"], motion["2019-01-23"]

## 2. Why a straight line isn't enough
Three days ≈ 72 hours at ~37 "/hr ≈ 2,700" (0.75°). The path curves a little, so we allow a tolerance proportional to the
distance travelled, and require similar velocities on both nights.

In [ ]:
pairs = link_tracklets_across_nights(first, second)
named = (pairs.merge(first[["tracklet_id", "known_name"]].rename(columns={"tracklet_id": "first_id", "known_name": "night_1"}), on="first_id")
              .merge(second[["tracklet_id", "known_name"]].rename(columns={"tracklet_id": "second_id", "known_name": "night_2"}), on="second_id"))
both = set(first.loc[first["known_name"] != "", "known_name"]) & set(second.loc[second["known_name"] != "", "known_name"])
correct = int(((named["night_1"] == named["night_2"]) & (named["night_1"] != "")).sum())
print(f"known asteroids on both nights: {len(both)}; linked correctly: {correct}; wrong pairs: {int((named['night_1'] != named['night_2']).sum())}")
named[["night_1", "night_2", "separation_arcsec", "tolerance_arcsec", "velocity_difference"]].round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for table, marker, label in [(first, "o", "2019-01-20"), (second, "s", "2019-01-23")]:
    ax.scatter(table["ra"], table["dec"], marker=marker, label=label)
for row in named.itertuples():
    a = first.set_index("tracklet_id").loc[row.first_id]; b = second.set_index("tracklet_id").loc[row.second_id]
    ax.plot([a["ra"], b["ra"]], [a["dec"], b["dec"]], "k-", lw=0.8)
ax.invert_xaxis(); ax.set_xlabel("RA (deg)"); ax.set_ylabel("Dec (deg)"); ax.legend(); ax.set_title("linked across 3 days")
plt.show()

## 3. The catch for real discoveries
In 3 days a main-belt asteroid moves ~0.75°, almost a whole CCD quadrant (0.87°). Most night-1 objects are in a
**neighbouring quadrant** on night 2, so a production system links across all quadrants of a field (or several fields).

## Exercises
1. Set `velocity_tolerance=0.25`. Which wrong pair appears, and what gives it away?
2. Which unknown night-1 tracklets would you expect to land outside this quadrant on night 2? (Use their v_ra, v_dec.)
3. Why is a two-night arc still not enough to compute a reliable orbit?